# 02 - Preprocessing

Section **3.3 Preprocessing** of the report.

Builds two model-ready datasets from the Wine Reviews v2 CSV (regression target: `points`):

- `data/processed/xgboost/` — unscaled tabular features for notebook 03
- `data/processed/nn/` — scaled tabular + TF-IDF text for notebook 04

No model training is performed here.

## Brief

- Drop redundant / structural columns (`region_2`, `taster_twitter_handle`, raw `title` after vintage extraction).
- Engineer numeric and binary features; target-encode high-cardinality fields (train-only CV).
- Stratified train / validation / test split on binned `points`.
- Export **two** processed datasets (XGBoost unscaled, NN scaled continuous + TF-IDF).
- Persist encoders, scaler, manifest under `data/processed/`.

In [2]:
import json

import numpy as np
import pandas as pd
from IPython.display import display

from diplo_mod_1.constants import INTERIM, PROCESSED, RANDOM_STATE, RAW
from diplo_mod_1.preprocessing import (
    add_base_features,
    build_encoded_splits,
    build_nn_tabular,
    clean_dataframe,
    export_nn_dataset,
    export_xgboost_dataset,
    fit_tfidf,
    load_raw,
    scale_continuous,
    stratified_split,
    write_manifest,
)

## Step 1 — Cleaning

Drop `region_2` and `taster_twitter_handle`, extract `vintage_year` from `title`, impute `price` (median by `country` + `variety`) and key categoricals.

In [3]:
df = load_raw(RAW / "winemag-data-130k-v2.csv")
print(f"Raw shape: {df.shape}")

cleaned, config = clean_dataframe(df)
INTERIM.mkdir(parents=True, exist_ok=True)
cleaned.to_parquet(INTERIM / "01_cleaned.parquet")
(INTERIM / "preprocessing_config.json").write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)
print("Missing after clean (top):")
print(cleaned.isnull().sum().sort_values(ascending=False).head(8))

Raw shape: (129971, 13)
Missing after clean (top):
designation     37465
vintage_year     4626
province           63
description         0
points              0
country             0
price               0
region_1            0
dtype: int64


## Step 2 — Base features

Non-target features: `log_price`, flags (`is_luxury`, `is_us`, `has_designation`, `price_missing`), `wine_age`, `description_length`.

In [4]:
featured = add_base_features(cleaned, config)
featured.to_parquet(INTERIM / "02_features.parquet")
featured[["log_price", "wine_age", "is_luxury", "description_length"]].describe()

,log_price,wine_age,is_luxury,description_length
count,129971.000000,129971.000000,129971.000000,129971.000000
mean,3.353673,1997.013341,0.005247,242.601065
std,0.618422,0.114732,0.072248,66.584276
min,1.609438,1997.000000,0.000000,20.000000
25%,2.890372,1997.000000,0.000000,197.000000
50%,3.258097,1997.000000,0.000000,237.000000
75%,3.761200,1997.000000,0.000000,282.000000
max,8.101981,1998.000000,1.000000,829.000000


## Step 3 — Train / validation / test split

Stratified on binned `points` (~64% / 16% / 20%). Indices are shared by both exported datasets.

In [5]:
split_idx = stratified_split(featured, random_state=RANDOM_STATE)
PROCESSED.mkdir(parents=True, exist_ok=True)
np.savez(
    PROCESSED / "split_indices.npz",
    train=split_idx["train"],
    val=split_idx["val"],
    test=split_idx["test"],
)
for name, idx in split_idx.items():
    print(f"{name}: {len(idx):,} rows ({100 * len(idx) / len(featured):.1f}%)")

train: 83,180 rows (64.0%)
val: 20,796 rows (16.0%)
test: 25,995 rows (20.0%)


## Step 4 — Encode and export XGBoost dataset

Target encoding (`TargetEncoder`, `target_type='continuous'`, cv=5 on train), frequency maps, OHE for `taster_name` and `country`. Matrix is **unscaled**.

In [6]:
x_xgb, y_splits, artifacts = build_encoded_splits(featured, split_idx)
xgb_dir = PROCESSED / "xgboost"
export_xgboost_dataset(xgb_dir, x_xgb, y_splits, artifacts)

meta = json.loads((xgb_dir / "feature_names.json").read_text(encoding="utf-8"))
print(f"XGBoost features: {len(meta['feature_names'])}")
print(f"X_train shape: {x_xgb['train'].shape}")
print("First 15 feature names:", meta["feature_names"][:15])

XGBoost features: 15
X_train shape: (83180, 15)
First 15 feature names: ['log_price', 'wine_age', 'description_length', 'price_missing', 'vintage_missing', 'is_luxury', 'is_us', 'has_designation', 'taster_avg_points', 'variety_avg_points', 'country_avg_points', 'region1_avg_points', 'winery_avg_points', 'winery_freq', 'variety_review_count']


## Step 5 — Export NN dataset

Tabular branch: add `taster_strictness`, scale continuous columns with `StandardScaler` (fit on train). Text branch: TF-IDF on `description` (2000 features, fit on train).

In [7]:
x_nn_raw, nn_names, nn_groups = build_nn_tabular(featured, split_idx, artifacts)
x_nn_scaled, scaler = scale_continuous(x_nn_raw, nn_groups["continuous"])

train_df = featured.iloc[split_idx["train"]]
val_df = featured.iloc[split_idx["val"]]
test_df = featured.iloc[split_idx["test"]]
x_txt, vectorizer = fit_tfidf(train_df, val_df, test_df)

nn_dir = PROCESSED / "nn"
export_nn_dataset(
    nn_dir,
    x_nn_scaled,
    x_txt,
    y_splits,
    nn_names,
    nn_groups,
    scaler,
    vectorizer,
)

print(f"NN tabular features: {len(nn_names)}")
print(f"X_tab_train shape: {x_nn_scaled['train'].shape}")
print(f"X_txt_train shape: {x_txt['train'].shape}, nnz={x_txt['train'].nnz:,}")

NN tabular features: 15
X_tab_train shape: (83180, 15)
X_txt_train shape: (83180, 2000), nnz=1,950,143


## Step 6 — Dataset manifest

Single JSON contract for notebooks 03–05.

In [8]:
write_manifest(
    PROCESSED / "dataset_manifest.json",
    xgb_dir,
    nn_dir,
    PROCESSED / "split_indices.npz",
    {k: x_xgb[k].shape for k in x_xgb},
    {k: x_nn_scaled[k].shape for k in x_nn_scaled},
    {k: x_txt[k].nnz for k in x_txt},
)
manifest = json.loads((PROCESSED / "dataset_manifest.json").read_text(encoding="utf-8"))
display(manifest)

{'target': 'points',
 'random_state': 42,
 'split': 'split_indices.npz',
 'xgboost': {'path': 'xgboost/',
  'scaled': False,
  'X_shape_train': [83180, 15],
  'X_shape_val': [20796, 15],
  'X_shape_test': [25995, 15]},
 'nn': {'path': 'nn/',
  'scaled': 'continuous_only',
  'X_tab_shape_train': [83180, 15],
  'X_tab_shape_val': [20796, 15],
  'X_tab_shape_test': [25995, 15],
  'X_txt_nnz_train': 1950143,
  'X_txt_nnz_val': 487458,
  'X_txt_nnz_test': 608803},
 'note': 'Training not performed in notebook 02.'}

## Design notes (report)

**EDA-driven decisions (notebook 01):**

- `region_2` dropped (61% missing, structural); `is_us` retains geographic signal.
- `log_price` and `is_luxury` handle extreme price skew (Spearman ρ ≈ 0.61 with `points`).
- `taster_avg_points` (target-encoded) captures scorer bias (ANOVA η² ≈ 0.10).
- `vintage_year` dropped from feature matrix — perfectly collinear with `wine_age`; only `wine_age` is kept.
- `taster_name` and `country` removed from OHE — already represented as target-encoded floats, OHE would duplicate the signal.
- High-cardinality fields (`winery`, `variety`) use target / frequency encoding, not full one-hot.

**Leakage controls:** all encoders, frequency maps, TF-IDF, and scaler fit on **train** only.

**Two datasets:** XGBoost needs unscaled inputs; the NN needs normalized continuous columns and sparse text features. Training happens in notebooks 03 and 04 respectively.